# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


### Unit of analysis
One row represents the daily performance of one content item for one client.

### Grain key
`report_date` + `client_hash_id` + `content_hash_id`

### Time window
March 1–31, 2026. This is the development partition used for contract verification, not the final holdout set.

### Intended use
Define a trustworthy feature dataset for content-performance analysis while respecting source availability and preventing data leakage.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import duckdb

# Connect to the March 2026 partition only
con = duckdb.connect()

import getpass

token = os.environ.get("HF_TOKEN")

if not token:
    token = getpass.getpass("Enter your Hugging Face token: ")

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [token]
)

data_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("DuckDB:", duckdb.__version__)
print("Target partition: March 2026")
print("Connection ready:", con is not None)


DuckDB: 1.5.5
Target partition: March 2026
Connection ready: True


In [11]:

# Test that the March 2026 partition is readable
result = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(?)
""", [data_path]).fetchdf()

result


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [12]:

schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [data_path]).fetchdf()

schema[["column_name", "column_type"]]



,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:

# Verification query 1: Check for duplicate grain keys
duplicate_keys = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS records_per_key
    FROM read_parquet(?)
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""", [data_path]).fetchdf()

print("Duplicate grain keys found:", len(duplicate_keys))
duplicate_keys



Duplicate grain keys found: 0


,report_date,client_hash_id,content_hash_id,records_per_key


In [14]:

# Verification query 2: Row count, date range, and data availability
summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(?)
""", [data_path]).fetchdf()

summary


,total_rows,first_date,last_date,gsc_available_rows,ga4_available_rows
0,9841378,2026-03-01,2026-03-31,3611061,413966


In [15]:

# Verification query 3: Count rows with both sources available
availability_check = con.execute("""
    SELECT
        COUNT(*) AS eligible_rows
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""", [data_path]).fetchdf()

availability_check


,eligible_rows
0,364347


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Feature fields 

The following fields are candidate predictors from the daily performance table:

1. `gsc_impressions` — potentially knowable after the daily GSC data for `report_date` has been finalized.
2. `gsc_clicks` — potentially knowable after the daily GSC data for `report_date` has been finalized.
3. `ga4_pageviews` — potentially knowable after the daily GA4 data for `report_date` has been finalized.
4. `ga4_sessions` — potentially knowable after the daily GA4 data for `report_date` has been finalized.
5. `ga4_engaged_sessions` — potentially knowable after the daily GA4 data for `report_date` has been finalized.

**Decision moment:** After the daily data for `report_date` has been finalized.

These are candidates, not yet approved predictors. Actual data availability and refresh timing must be verified. Source-specific availability flags must be respected, and future-period values must not enter the feature frame.

### Label or proxy
- `future_decline` — the intended binary target: whether a content item experiences a meaningful decline over the following 28 days.

The target has not yet been constructed or validated in this daily table. A defensible decline threshold and future observation window must be defined before training.

### Context fields
- `report_date`: establishes the observation date.
- `client_hash_id`: supports grouping and client-aware validation.
- `content_hash_id`: identifies content for grouping and tracking.
- `gsc_data_available`, `ga4_data_available`: indicate source availability.
- `client_has_gsc`, `client_has_ga4`: indicate whether the client has each source.

Context and identifiers must not be used as predictive features without a justified, leakage-safe design.

### Excluded fields
- `trend_direction` and `trend_pct` from the starter dataset: excluded because they are related to the existing trend label and may leak the outcome.
- `client_hash_id` and `content_hash_id`: excluded as direct model inputs because they are identifiers.
- Any future-window measurements: excluded from predictors because they would reveal information unavailable at decision time.
- GA4 fields for rows where `ga4_data_available IS NOT TRUE`; likewise, GSC fields where `gsc_data_available IS NOT TRUE`. Unavailable values must not be treated as genuine zeros.


In [16]:

# Build a March 2026 feature frame using at most five features
feature_frame = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""", [data_path]).fetchdf()

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

print("Feature frame shape:", feature_frame.shape)
print("Number of features:", len(feature_columns))
print("Date range:",
      feature_frame["report_date"].min(),
      "to",
      feature_frame["report_date"].max())

feature_frame[feature_columns].describe().T


Feature frame shape: (364347, 8)
Number of features: 5
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,count,mean,std,min,25%,50%,75%,max
gsc_impressions,364347.0,234.218259,519.789055,1.0,33.0,98.0,249.0,39305.0
gsc_clicks,364347.0,1.084691,3.079481,0.0,0.0,0.0,1.0,274.0
ga4_pageviews,364347.0,3.891389,8.425871,0.0,1.0,2.0,4.0,875.0
ga4_sessions,364347.0,3.399932,7.815113,0.0,1.0,1.0,3.0,792.0
ga4_engaged_sessions,364347.0,0.076776,0.314674,0.0,0.0,0.0,0.0,21.0



### When are these features knowable?

- `gsc_impressions`: Knowable after the daily Search Console data for the observation date has been finalized.
- `gsc_clicks`: Knowable after the daily Search Console data for the observation date has been finalized.
- `ga4_pageviews`: Knowable after the daily Analytics data for the observation date has been finalized.
- `ga4_sessions`: Knowable after the daily Analytics data for the observation date has been finalized.
- `ga4_engaged_sessions`: Knowable after the daily Analytics data for the observation date has been finalized.

These timing assumptions must be confirmed against the actual data refresh process. These are candidate predictors for a future outcome, not evidence that a validated prediction target already exists.


In [24]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Use the actual target column in the starter dataset
target_col = "trend_direction"

# Exclude target-related fields from the honest feature set
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
]

data = starter.dropna(subset=[target_col]).copy()

X = data[candidate_features].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median()).fillna(0)
y = data[target_col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Honest baseline: no trend_pct feature
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))

# Deliberate leakage: add the trend percentage used to derive trend direction
X_leaked = X.copy()
X_leaked["trend_pct"] = pd.to_numeric(
    data["trend_pct"], errors="coerce"
).fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaked, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaked_model = DecisionTreeClassifier(random_state=42)
leaked_model.fit(X_train_l, y_train_l)
leaked_score = accuracy_score(
    y_test_l, leaked_model.predict(X_test_l)
)

print(f"Honest baseline accuracy: {honest_score:.3f}")
print(f"Accuracy with leaked feature: {leaked_score:.3f}")
print("Leakage feature: trend_pct")
print("Target: trend_direction")


Honest baseline accuracy: 0.667
Accuracy with leaked feature: 1.000
Leakage feature: trend_pct
Target: trend_direction



### Leakage experiment

The experiment used the project's anonymized starter dataset to demonstrate target leakage.

- Target: `trend_direction`
- Honest baseline accuracy: 66.7%.
- Accuracy with the leaked feature: 100.0%.
- Leaked feature: `trend_pct`.

The leaked feature is related to the construction of `trend_direction`, allowing the model to infer the target directly. The perfect score is therefore misleading and does not represent valid predictive performance.

**Conclusion:** `trend_pct` must be excluded from predictors when predicting `trend_direction`.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


### Data limitations

1. **Source availability:** Only 3,611,061 of 9,841,378 March records have GSC data available, and 413,966 have GA4 data available. Only 364,347 records have both sources available.

2. **Missing data versus zero:** Unavailable source data must not be interpreted as genuine zero activity. Feature eligibility must respect the corresponding availability flags.

3. **Temporal scope:** March 2026 is the development partition used for this contract. The June 2026 partition is reserved as the final holdout and must not be used for development decisions.

4. **Target definition:** The daily performance table does not contain a validated future-decline label. A target definition and future observation window must be established before training a predictive model.

5. **Leakage risk:** Predictors must contain only information available at the intended prediction time. Future-period measurements and label-derived fields must be excluded.

6. **Generalization:** Results from one month may not generalize to other months or clients. Temporal and client-level validation should be considered.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.